In [11]:
import pandas as pd
import json
import re
import pypdf
import sys

def load_users(file_path):
    """Loads user data from a JSON file."""
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        return pd.DataFrame(data)
    except FileNotFoundError:
        print(f"Error: Could not find user file at {file_path}")
        sys.exit(1)

def parse_restaurants_sql(file_path):
    """Parses raw SQL INSERT statements to extract restaurant data."""
    try:
        with open(file_path, 'r') as f:
            sql_text = f.read()
            
        # Regex to capture (id, name, cuisine, rating)
        # Looks for: VALUES (123, 'Name', 'Cuisine', 4.5);
        pattern = r"VALUES\s*\((\d+),\s*'([^']*)',\s*'([^']*)',\s*([\d\.]+)\);"
        matches = re.findall(pattern, sql_text)
        
        df = pd.DataFrame(matches, columns=['restaurant_id', 'restaurant_name', 'cuisine', 'rating'])
        
        # Ensure numbers are actual numbers, not strings
        df['restaurant_id'] = pd.to_numeric(df['restaurant_id'])
        df['rating'] = pd.to_numeric(df['rating'])
        
        return df
    except FileNotFoundError:
        print(f"Error: Could not find SQL file at {file_path}")
        sys.exit(1)

def extract_orders_from_pdf(pdf_path):
    """Reads tables from the PDF and converts them to a DataFrame."""
    try:
        reader = pypdf.PdfReader(pdf_path)
        full_text = ""
        for page in reader.pages:
            full_text += page.extract_text() + "\n"
            
        lines = full_text.split('\n')
        parsed_data = []
        
        # Regex for the specific PDF row format
        # Group 1: OrderID, 2: UserID, 3: RestID, 4: Date, 5: Amount
        row_regex = re.compile(r"^(\d+)\s+(\d+)\s+(\d+)\s+(\d{2}-\d{2}-\d{4})\s*([\d\.]+)")
        
        for line in lines:
            cleaned_line = line.strip()
            match = row_regex.match(cleaned_line)
            if match:
                parsed_data.append({
                    'order_id': int(match.group(1)),
                    'user_id': int(match.group(2)),
                    'restaurant_id': int(match.group(3)),
                    'order_date': match.group(4),
                    'total_amount': float(match.group(5))
                })
                
        return pd.DataFrame(parsed_data)
        
    except FileNotFoundError:
        print(f"Error: Could not find PDF file at {pdf_path}")
        sys.exit(1)
    except Exception as e:
        print(f"An error occurred parsing the PDF: {e}")
        sys.exit(1)

def generate_insights(df):
    """Calculates and prints the required analytics."""
    print("\n" + "="*30)
    print("📊 DATA ANALYSIS REPORT")
    print("="*30)

    # 1. Gold Member Revenue by City
    gold_rev = df[df['membership'] == 'Gold'].groupby('city')['total_amount'].sum()
    top_city = gold_rev.idxmax()
    print(f"1. Top City (Gold Revenue):  {top_city}")

    # 2. Cuisine Analysis
    cuisine_aov = df.groupby('cuisine')['total_amount'].mean()
    print(f"2. Highest AOV Cuisine:      {cuisine_aov.idxmax()} (₹{cuisine_aov.max():.2f})")

    # 3. High Value Users
    user_spend = df.groupby('user_id')['total_amount'].sum()
    high_rollers = (user_spend > 1000).sum()
    print(f"3. Users spending > ₹1k:     {high_rollers}")

    # 4. Top Rating Tier
    # Using a lambda for cleaner categorization
    df['rating_band'] = df['rating'].apply(lambda x: 
        '4.6-5.0' if x >= 4.6 else 
        '4.1-4.5' if x >= 4.1 else 
        '3.6-4.0' if x >= 3.6 else '3.0-3.5')
    
    top_band = df.groupby('rating_band')['total_amount'].sum().idxmax()
    print(f"4. Most Profitable Ratings:  {top_band}")

    # 5. Top Quarter
    # Ensure date is datetime format
    df['quarter'] = df['order_date'].dt.quarter
    best_q = df.groupby('quarter')['total_amount'].sum().idxmax()
    print(f"5. Best Quarter:             Q{best_q}")

def main():
    # File Configuration
    files = {
        'users': 'users.json',
        'restaurants': 'restaurants.sql',
        'orders': 'Copy of orders - orders.pdf'
    }

    # 1. Load Data
    print("...Loading datasets...")
    df_users = load_users(files['users'])
    df_rest = parse_restaurants_sql(files['restaurants'])
    df_orders = extract_orders_from_pdf(files['orders'])

    # 2. Merge Data
    print("...Merging tables...")
    # Left join orders -> users
    merged = pd.merge(df_orders, df_users, on='user_id', how='left')
    
    # Left join result -> restaurants
    # Note: 'suffixes' handles column name collisions automatically
    final_dataset = pd.merge(merged, df_rest, on='restaurant_id', how='left', suffixes=('_trans', '_master'))

    # Convert date column once for all analysis
    final_dataset['order_date'] = pd.to_datetime(final_dataset['order_date'], format='%d-%m-%Y')

    # 3. Save Output
    output_file = 'final_food_delivery_dataset.csv'
    final_dataset.to_csv(output_file, index=False)
    print(f"✅ Success! Saved {len(final_dataset)} rows to '{output_file}'")

    # 4. Run Analysis
    generate_insights(final_dataset)

if __name__ == "__main__":
    main()

...Loading datasets...
...Merging tables...
✅ Success! Saved 10000 rows to 'final_food_delivery_dataset.csv'

📊 DATA ANALYSIS REPORT
1. Top City (Gold Revenue):  Chennai
2. Highest AOV Cuisine:      Mexican (₹808.02)
3. Users spending > ₹1k:     2544
4. Most Profitable Ratings:  4.6-5.0
5. Best Quarter:             Q3


In [3]:
!pip install pypdf